In [3]:
import pyarrow.parquet as pq
import os

pl.Config.set_tbl_cols(30)          # Show up to 20 columns
pl.Config.set_tbl_rows(80)   
pl.Config.set_tbl_width_chars(500)  # Make the table wider in the console
pl.Config.set_fmt_str_lengths(50)   # Don't cut off long strings like stoch_key


# Define the file path
file_path = r'C:\Users\Owner\airflow-trading\data_lake\Saved_results\Second_CCD_run\equity_search_partitioned\_tmp\equity_era_int=20220901_batch=0_task=0.parquet'

# Check if file exists to avoid errors
if os.path.exists(file_path):
    # Open the parquet file metadata
    parquet_file = pq.ParquetFile(file_path)
    
    # 1. Get the Schema
    schema = parquet_file.schema.to_arrow_schema()
    
    # 2. Get Total Row Count
    total_rows = parquet_file.metadata.num_rows
    
    print(f"--- File Statistics ---")
    print(f"Total Rows: {total_rows:,}")
    print(f"\n--- PyArrow Schema ---")
    print(schema)
else:
    print(f"Error: File not found at {file_path}")

--- File Statistics ---
Total Rows: 3,030

--- PyArrow Schema ---
regime_id: int32
era_int: int64
side: int8
SL: float
TP: float
time_ns: int64
entry_idx: int64
exit_idx: int64
pnl_pct: float
equity: float


In [4]:
import polars as pl
import os

# Define the directory path
directory = r'C:\Users\Owner\airflow-trading\data_lake\Saved_results\Second_CCD_run\equity_search_partitioned\_tmp'
# Pattern to match your task/batch files
path_pattern = os.path.join(directory, "*.parquet")

# Read all files into a single RAM variable
# Polars handles the vertical concatenation automatically
df = pl.read_parquet(path_pattern)

print(f"Total rows loaded: {len(df)}")
print(df.head())

Total rows loaded: 41437
shape: (5, 10)
┌───────────┬──────────┬──────┬─────┬─────┬─────────────────────┬───────────┬──────────┬───────────┬────────────┐
│ regime_id ┆ era_int  ┆ side ┆ SL  ┆ TP  ┆ time_ns             ┆ entry_idx ┆ exit_idx ┆ pnl_pct   ┆ equity     │
│ ---       ┆ ---      ┆ ---  ┆ --- ┆ --- ┆ ---                 ┆ ---       ┆ ---      ┆ ---       ┆ ---        │
│ i32       ┆ i64      ┆ i8   ┆ f32 ┆ f32 ┆ i64                 ┆ i64       ┆ i64      ┆ f32       ┆ f32        │
╞═══════════╪══════════╪══════╪═════╪═════╪═════════════════════╪═══════════╪══════════╪═══════════╪════════════╡
│ 0         ┆ 20220901 ┆ 1    ┆ 0.3 ┆ 5.0 ┆ 1662034800000000000 ┆ 4430      ┆ 4468     ┆ 0.011667  ┆ 101.166672 │
│ 0         ┆ 20220901 ┆ 1    ┆ 0.3 ┆ 5.0 ┆ 1662110400000000000 ┆ 4710      ┆ 4720     ┆ -0.001833 ┆ 100.981194 │
│ 0         ┆ 20220901 ┆ 1    ┆ 0.3 ┆ 5.0 ┆ 1662186000000000000 ┆ 4970      ┆ 4972     ┆ -0.006333 ┆ 100.341652 │
│ 0         ┆ 20220901 ┆ 1    ┆ 0.3 ┆ 5.0 ┆ 1662

In [ ]:
# Returns the number of unique SL+TP pairs
unique_count = df.select(pl.col(["SL"])).n_unique()

print(f"Total Unique SL+TP Combinations: {unique_count}")

Total Unique SL+TP Combinations: 2
